# Классификатор на основе формулы Байеса

Поскольку задача состоит в восстановлении для заданного набора нуликов и единичек другого набора нуликов и единичек, то возникают естественные аналогии с вероятностью и формулой Байеса, смысл которой как раз и заключается в нечетком выводе «что же там на самом деле было».

Пусть \\(X^0=(x_{ij}^0)\\) — входная матрица, для которой необходимо произвести восстановление \\(Y=(y_{ij})\\). Нас интересует вероятность того, что \\(y_{ij} = 1\\) при том, что \\(x_{ij} = x_{ij}^0\\), т.е.
\\[
p_{ij} \triangleq P(y_{ij} = 1 | X = X^0).
\\]
Если мы определим эту вероятность для всех пар индексов \\(i\\) и \\(j\\), то
матрицу \\(P = (p_{ij})\\) можно считать приближенным ответом, так как чем больше \\(p_{ij}\\), тем больше вероятность того, что \\(y_{ij} = 1\\), и, наоборот, чем меньше \\(p_{ij}\\), тем вероятнее, что \\(y_{ij} = 0\\).

По формуле Байеса
\\[
P(y_{ij} = 1 | X = X^0) = \frac{P(y_{ij} = 1) P(X = X^0 | y_{ij} = 1)}{P(y_{ij} = 1) P(X = X^0 | y_{ij} = 1) + P(y_{ij} = 0) P(X = X^0 | y_{ij} = 0)}.
\tag{B}
\\]

Выражения в правой части (B) будем приближенно вычислять на основе имеющихся наборов входов \\(x_{train}\\) и правильных ответов \\(y_{train}\\).

Для заданного входа \\(X^0=(x_{ij}^0)\\), вычислив матрицу \\(P = (p_{ij})\\), ответом будем считать выражение \\([P]\\), где \\([\cdot]\\) — операция математического округления, применяемая поэлементно.

## Импортирование необходимых библиотек

In [ ]:
import pandas, numpy
import os
import matplotlib.pyplot as plot
from tqdm import tqdm

## Подготовка данных

In [ ]:
data_train = pandas.read_csv(os.path.join('/', 'kaggle', 'input', 'ozon-masters-2020', 'gtrain.csv'))
data_test = pandas.read_csv(os.path.join('/', 'kaggle', 'input', 'ozon-masters-2020', 'gtest.csv'))

Разделим тестовую и обучающую выборки на группы индексов по числу шагов.

In [ ]:
nsteps_indices = []
test_nsteps_indices = []

for k in range(1, 6):
    nsteps_indices.append(numpy.array(data_train[data_train.steps == k].index))
    test_nsteps_indices.append(numpy.array(data_test[data_test.steps == k].index))

In [ ]:
N, N2 = 22, 484

SIZE = (data_train.shape[0], N, N)

y_train = data_train.iloc[:, 2+N2:2+2*N2].to_numpy().reshape(SIZE)
x_train = data_train.iloc[:, 2:2+N2].to_numpy().reshape(SIZE)

TSIZE = (data_test.shape[0], N, N)

x_test = data_test.iloc[:, 2:2+N2].to_numpy().reshape(TSIZE)
y_test = numpy.empty(TSIZE, dtype = 'float32')

## Магический квадрат и его братья-близнецы
Для того чтобы вычислить вероятность \\(P(y_{ij} = 1)\\) появления \\(1\\) в позиции \\((i, j)\\) у случайной величины \\(Y\\) проведем усреднение имеющихся в нашем распоряжении правильных ответов. В том, что вероятности \\(P(y_{ij} = 1)\\) действительно определяются этим усреднением, мы убедимся позднее.

In [ ]:
magic_square = numpy.mean(y_train, axis = 0)

plot.imshow(magic_square)
plot.title('Магический квадрат')
plot.show()

Отсюда видно, что вероятность появления \\(1\\) на побочных диагоналях «магического квадрата» примерно одинакова и эта вероятность возрастает от северо-западного угла к юго-восточному. Нетрудно убедиться, что характер магического квадрата несильно меняется, если мы усредняем правильные ответы только по конкретному количеству шагов \\(s\\). Разумеется, \\(P(y_{ij} = 0) = 1 - P(y_{ij} = 1)\\).

Давайте построим братьев-близнецов магического квадрата, получаемых усреднением входов \\(x_{train}\\) и \\(x_{test}\\) соответственно.

In [ ]:
magic_square_brer_train = numpy.mean(x_train, axis = 0)
magic_square_brer_test = numpy.mean(x_test, axis = 0)

f, axes = plot.subplots(1, 2)

axes[0].imshow(magic_square_brer_train.reshape(N, N))
axes[0].set_title('Брат магического квадрата')

axes[1].imshow(magic_square_brer_test.reshape(N, N))
axes[1].set_title('Его близнец')

plot.show()

Как мы видим, они действительно похожи. Следовательно, можно предположить, что набор \\(y_{test}\\), который необходимо предсказать, устроен так, что его среднее близко к магическому квадрату. Это дает нам полные основания применять байесовский подход, считая, что вероятности \\(P(y_{ij} = 1)\\) определяются из магического квадрата.

## Байесовский классификатор в действии

В предположениях независимости, вероятность \\(P(X = X^0 | y_{ij} = 1)\\) представим в виде совместного появления событий \\(x_{st} = x_{st}^0\\) при
\\(y_{ij} = 1\\). Тогда
\\[
P(X = X^0 | y_{ij} = 1)
=\prod_{s, t = 1}^{22}
P(x_{st} = x_{st}^0 | y_{ij} = 1).
\\]
Выражение \\(P(x_{st} = x_{st}^0 | y_{ij} = 1)\\) вычисляем следующим образом:
1. запоминаем все индексы, для которых \\(y_{train}\\) в позиции \\((i, j)\\) имеет значение \\(1\\);
1. проводим усреднение набора \\(x_{train}\\) по указанным индексам, которое обозначим через \\(AVG[x_{train} | y_{train}[i, j] = 1]\\);
1. если \\(x_{st}^0 = 1\\), то положим
\\[
P(x_{st} = x_{st}^0 | y_{ij} = 1) = AVG[x_{train} | y_{train}[i, j] = 1]_{st}.
\\]
В противном случае —
\\[
P(x_{st} = x_{st}^0 | y_{ij} = 1) = 1 - AVG[x_{train} | y_{train}[i, j] = 1]_{st}.
\\]

Выражение \\(P(X = X^0 | y_{ij} = 0)\\) вычисляется аналогично.

Разумеется, все усреднения проводятся по подиндексам, отвечающим заданному числу шагов.

In [ ]:
def bayes_predict(image, steps_indices, nsteps_range = range(5)):
    ONE_CP = numpy.empty((N, N, N, N), dtype = 'float32')
    ZERO_CP = numpy.empty((N, N, N, N), dtype = 'float32')
    result = numpy.empty((N, N), dtype = 'float32')

    for nsteps_ in nsteps_range:
        indices = steps_indices[nsteps_]

        nstep_x_train = x_train[nsteps_indices[nsteps_]]
        nstep_y_train = y_train[nsteps_indices[nsteps_]]

        items2pred = image[indices]

        # Для заданного числа шагов формируем матрицы условных вероятностей.
        with tqdm(total = N2, desc = 'Число шагов = %d -> Подготовка' % (nsteps_ + 1), bar_format = "{desc}:   [ осталось: {remaining}; прошло: {elapsed} ] {percentage:3.0f}%|{bar}") as pbar:
            for i in range(N):
                for j in range(N):
                    ZERO_CP[i, j, :] = numpy.mean(nstep_x_train[nstep_y_train[:, i, j] == 0], axis = 0)
                    ONE_CP[i, j, :] = numpy.mean(nstep_x_train[nstep_y_train[:, i, j] == 1], axis = 0)
                    ######################################
                    pbar.update(1)

        # Предсказываем...
        with tqdm(total = items2pred.shape[0], desc = 'Число шагов = %d -> Предсказание' % (nsteps_ + 1), bar_format = "{desc}: [ осталось: {remaining}; прошло: {elapsed} ] {percentage:3.0f}%|{bar}") as pbar:
            for k in range(items2pred.shape[0]):

                item2pred = items2pred[k]
                inverted = item2pred ^ 1

                for i in range(N):
                    for j in range(N):
                        c = (ONE_CP[i, j] * item2pred + inverted * (1 - ONE_CP[i, j])).prod()
                        d = (ZERO_CP[i, j] * item2pred + inverted * (1 - ZERO_CP[i, j])).prod()

                        v = magic_square.item(i, j)

                        # В числителе намеренно не умножаем на magic_square.itemset(i, j), поскольку так медленнее.
                        result.itemset(i, j, c / (v * c + (1 - v) * d))

                # Умножаем сразу всю матрицу здесь.
                result *= magic_square

                y_test[indices[k], :] = result
                ######################################
                pbar.update(1)

In [ ]:
bayes_predict(x_test, test_nsteps_indices)

## Формирование ответа

In [ ]:
columns = []

for k in range(N2):
    columns.append(f'y_{k}')
    
answer = numpy.int32(y_test + .5).reshape(data_test.shape[0], N2)

submission = pandas.DataFrame(answer, columns = columns)

submission.insert(0, 'id', data_test['id'].values)
submission.to_csv('submission.csv', index = False)